### Movie Data Wrangling

This notebook inspects, cleans, and prepares the datasets

In [242]:
from pathlib import Path
import pandas as pd

Load datasets from the `../data/raw` directory


In [243]:
raw_data_directory = Path("../data/raw")

#### Utility Functions

In [244]:
def inspect_dataset(dataset):
    print(f"Rows: {dataset.shape[0]:,}")
    print(f"Columns: {dataset.shape[1]}")

    print("\nDataset information:")
    dataset.info()

    print("\nFirst five rows:")
    display(dataset.head())

In [245]:
def summarize_missing_values(dataset):
    missing_count = dataset.isna().sum()
    missing_percentage = dataset.isna().mean() * 100

    summary = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percentage": missing_percentage.round(2)
    })

    return summary.sort_values(
        by="missing_percentage",
        ascending=False
    )

In [246]:
def inspect_missing_records(dataset, column_name):
    missing_records = dataset[dataset[column_name].isna()]

    print(f"Missing {column_name}: {len(missing_records):,}")

    display(
        missing_records[
            ["Year", "Category", "Film", "Name", "Nominees"]
        ].head(10)
    )

In [247]:
def save_cleaned_data(df, filename):
    output_dir = Path("../data/processed")
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / filename
    df.to_csv(output_path, index=False)

    print(f"Saved to {output_path}")

#### Oscar data

In [248]:
oscars = pd.read_csv(raw_data_directory / "oscars.csv")

inspect_dataset(oscars)

Rows: 12,137
Columns: 14

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12137 entries, 0 to 12136
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Ceremony           12137 non-null  int64 
 1   Year               12137 non-null  object
 2   Class              12137 non-null  object
 3   CanonicalCategory  12137 non-null  object
 4   Category           12137 non-null  object
 5   Film               10857 non-null  object
 6   FilmId             10856 non-null  object
 7   Name               10934 non-null  object
 8   Nominees           11779 non-null  object
 9   NomineeIds         11252 non-null  object
 10  Winner             3515 non-null   object
 11  Detail             3187 non-null   object
 12  Note               626 non-null    object
 13  Citation           1201 non-null   object
dtypes: int64(1), object(13)
memory usage: 1.3+ MB

First five rows:


,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,NomineeIds,Winner,Detail,Note,Citation
0,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Noose|The Patent Leather Kid,tt0019217|tt0018253,Richard Barthelmess,Richard Barthelmess,nm0001932,NaN,Nickie Elkins|The Patent Leather Kid,NaN,NaN
1,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Last Command|The Way of All Flesh,tt0019071|tt0019553,Emil Jannings,Emil Jannings,nm0417837,True,General Dolgorucki [Grand Duke Sergius Alexand...,NaN,NaN
2,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,nm0237571,NaN,Mrs. Pleznik,NaN,NaN
3,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,7th Heaven|Street Angel|Sunrise,tt0018379|tt0019429|tt0018455,Janet Gaynor,Janet Gaynor,nm0310980,True,Diane|Angela|The Wife,NaN,NaN
4,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,Sadie Thompson,tt0019344,Gloria Swanson,Gloria Swanson,nm0841797,NaN,Sadie Thompson,NaN,NaN


In [249]:
summarize_missing_values(oscars)

,missing_count,missing_percentage
Note,11511,94.84
Citation,10936,90.10
Detail,8950,73.74
Winner,8622,71.04
Film,1280,10.55
FilmId,1281,10.55
Name,1203,9.91
NomineeIds,885,7.29
Nominees,358,2.95
Category,0,0.00


In [250]:
oscars["Winner"].value_counts(dropna=False)

Winner
NaN     8622
True    3515
Name: count, dtype: int64

In [251]:
oscars_cleaned = oscars.copy()

oscars_cleaned["Winner"] = (
    oscars_cleaned["Winner"]
    .fillna(False)
    .astype(bool)
)

oscars_cleaned["Winner"].value_counts(dropna=False)

/var/folders/c5/26w6_6bj1tndp8sxfdckrkc80000gn/T/ipykernel_86491/3117872327.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


Winner
False    8622
True     3515
Name: count, dtype: int64

In [252]:
missing_film_titles = oscars_cleaned[
    oscars_cleaned["Film"].isna()
]

display(
    missing_film_titles.head()
)

,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,NomineeIds,Winner,Detail,Note,Citation
15,1,1927/28,Production,VISUAL EFFECTS,ENGINEERING EFFECTS,NaN,NaN,Ralph Hammeras,Ralph Hammeras,nm0358518,False,NaN,NOTE: This nomination was not associated with ...,NaN
17,1,1927/28,Production,VISUAL EFFECTS,ENGINEERING EFFECTS,NaN,NaN,Nugent Slaughter,Nugent Slaughter,nm0805521,False,NaN,NOTE: Though no specific titles were indicated...,NaN
30,1,1927/28,Writing,WRITING (Title Writing),WRITING (Title Writing),NaN,NaN,Joseph Farnham,Joseph Farnham,nm0267868,True,NaN,NOTE: This award was not associated with any s...,NaN
31,1,1927/28,Writing,WRITING (Title Writing),WRITING (Title Writing),NaN,NaN,"George Marion, Jr.",George Marion Jr.,nm0547938,False,NaN,NOTE: This nomination was not associated with ...,NaN
144,4,1930/31,Production,SOUND RECORDING,SOUND RECORDING,NaN,NaN,Samuel Goldwyn - United Artists Studio Sound D...,Samuel Goldwyn - United Artists,co0058013,False,NaN,NOTE: This nomination was not associated with ...,NaN


In [253]:
missing_film_ids = oscars_cleaned[
    oscars_cleaned["FilmId"].isna()
]

display(
    missing_film_ids.head()
)

,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,NomineeIds,Winner,Detail,Note,Citation
15,1,1927/28,Production,VISUAL EFFECTS,ENGINEERING EFFECTS,NaN,NaN,Ralph Hammeras,Ralph Hammeras,nm0358518,False,NaN,NOTE: This nomination was not associated with ...,NaN
17,1,1927/28,Production,VISUAL EFFECTS,ENGINEERING EFFECTS,NaN,NaN,Nugent Slaughter,Nugent Slaughter,nm0805521,False,NaN,NOTE: Though no specific titles were indicated...,NaN
30,1,1927/28,Writing,WRITING (Title Writing),WRITING (Title Writing),NaN,NaN,Joseph Farnham,Joseph Farnham,nm0267868,True,NaN,NOTE: This award was not associated with any s...,NaN
31,1,1927/28,Writing,WRITING (Title Writing),WRITING (Title Writing),NaN,NaN,"George Marion, Jr.",George Marion Jr.,nm0547938,False,NaN,NOTE: This nomination was not associated with ...,NaN
144,4,1930/31,Production,SOUND RECORDING,SOUND RECORDING,NaN,NaN,Samuel Goldwyn - United Artists Studio Sound D...,Samuel Goldwyn - United Artists,co0058013,False,NaN,NOTE: This nomination was not associated with ...,NaN


In [254]:
missing_film_ids = oscars_cleaned[
    oscars_cleaned["Film"].notna()
    & oscars_cleaned["FilmId"].isna()
]

print(f"Films without IMDb IDs: {len(missing_film_ids)}")

display(
    missing_film_ids.head(20)
)


Films without IMDb IDs: 1


,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,NomineeIds,Winner,Detail,Note,Citation
1587,16,1943,Title,DOCUMENTARY (Short Subject),DOCUMENTARY (Short Subject),Letter from Livingston,NaN,United States Army 4th Signal Photographic Unit,United States Army 4th Signal Photographic Unit,NaN,False,NaN,NOTE: THIS IS NOT AN OFFICIAL NOMINATION. Titl...,NaN


In [255]:
inspect_missing_records(oscars_cleaned, "Name")
inspect_missing_records(oscars_cleaned, "Nominees")

Missing Name: 1,203


,Year,Category,Film,Name,Nominees
32,1927/28,SPECIAL AWARD,The Jazz Singer,NaN,Warner Bros.
33,1927/28,SPECIAL AWARD,The Circus,NaN,Charles Chaplin
158,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class I),NaN,NaN,ELECTRICAL RESEARCH PRODUCTS INC.|RCA-PHOTOPHO...
159,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class I),NaN,NaN,DuPONT FILM MANUFACTURING CORP.|Eastman Kodak ...
160,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class II),NaN,NaN,Fox Film Corp.
161,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class III),NaN,NaN,Electrical Research Products Inc.
162,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class III),NaN,NaN,Rko Radio Pictures
163,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class III),NaN,NaN,Rca-Photophone Inc.
208,1931/32,SPECIAL AWARD,NaN,NaN,Walt Disney
209,1931/32,SCIENTIFIC OR TECHNICAL AWARD (Class II),NaN,NaN,Technicolor Motion Picture Corp.


Missing Nominees: 358


,Year,Category,Film,Name,Nominees
542,1936,SPECIAL AWARD,The March of Time,NaN,NaN
1166,1941,ART DIRECTION (Black-and-White),Sis Hopkins,NaN,NaN
1734,1944,ART DIRECTION (Black-and-White),Song of the Open Road,NaN,NaN
2265,1947,SPECIAL AWARD,Bill and Coo,NaN,NaN
2266,1947,SPECIAL AWARD,Shoe-Shine,NaN,NaN
2377,1948,SPECIAL FOREIGN LANGUAGE FILM AWARD,Monsieur Vincent,NaN,NaN
2495,1949,SPECIAL FOREIGN LANGUAGE FILM AWARD,The Bicycle Thief,NaN,NaN
2616,1950,HONORARY FOREIGN LANGUAGE FILM AWARD,The Walls of Malapaga,NaN,NaN
2738,1951,HONORARY FOREIGN LANGUAGE FILM AWARD,Rashomon,NaN,NaN
2871,1952,HONORARY FOREIGN LANGUAGE FILM AWARD,Forbidden Games,NaN,NaN


In [256]:
missing_recipients = oscars_cleaned[
    oscars_cleaned["Name"].isna()
    & oscars_cleaned["Nominees"].isna()
]

print(f"Records missing both fields: {len(missing_recipients):,}")

display(
    missing_recipients[
        ["Year", "Category", "Film", "Name", "Nominees"]
    ].head(20)
)

Records missing both fields: 13


,Year,Category,Film,Name,Nominees
542,1936,SPECIAL AWARD,The March of Time,NaN,NaN
1166,1941,ART DIRECTION (Black-and-White),Sis Hopkins,NaN,NaN
1734,1944,ART DIRECTION (Black-and-White),Song of the Open Road,NaN,NaN
2265,1947,SPECIAL AWARD,Bill and Coo,NaN,NaN
2266,1947,SPECIAL AWARD,Shoe-Shine,NaN,NaN
2377,1948,SPECIAL FOREIGN LANGUAGE FILM AWARD,Monsieur Vincent,NaN,NaN
2495,1949,SPECIAL FOREIGN LANGUAGE FILM AWARD,The Bicycle Thief,NaN,NaN
2616,1950,HONORARY FOREIGN LANGUAGE FILM AWARD,The Walls of Malapaga,NaN,NaN
2738,1951,HONORARY FOREIGN LANGUAGE FILM AWARD,Rashomon,NaN,NaN
2871,1952,HONORARY FOREIGN LANGUAGE FILM AWARD,Forbidden Games,NaN,NaN


In [257]:
inspect_missing_records(oscars_cleaned, "NomineeIds")

Missing NomineeIds: 885


,Year,Category,Film,Name,Nominees
158,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class I),NaN,NaN,ELECTRICAL RESEARCH PRODUCTS INC.|RCA-PHOTOPHO...
159,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class I),NaN,NaN,DuPONT FILM MANUFACTURING CORP.|Eastman Kodak ...
160,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class II),NaN,NaN,Fox Film Corp.
161,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class III),NaN,NaN,Electrical Research Products Inc.
162,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class III),NaN,NaN,Rko Radio Pictures
163,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class III),NaN,NaN,Rca-Photophone Inc.
209,1931/32,SCIENTIFIC OR TECHNICAL AWARD (Class II),NaN,NaN,Technicolor Motion Picture Corp.
210,1931/32,SCIENTIFIC OR TECHNICAL AWARD (Class III),NaN,NaN,Eastman Kodak Company
273,1932/33,SCIENTIFIC OR TECHNICAL AWARD (Class II),NaN,NaN,Electrical Research Products
274,1932/33,SCIENTIFIC OR TECHNICAL AWARD (Class II),NaN,NaN,Rca-Victor Company Inc.


In [258]:
missing_nominee_ids = oscars_cleaned[
    oscars_cleaned["NomineeIds"].isna()
]

display(
    missing_nominee_ids[
        ["Year", "Category", "Film", "Name", "Nominees", "NomineeIds"]
    ].head(10)
)

,Year,Category,Film,Name,Nominees,NomineeIds
158,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class I),NaN,NaN,ELECTRICAL RESEARCH PRODUCTS INC.|RCA-PHOTOPHO...,NaN
159,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class I),NaN,NaN,DuPONT FILM MANUFACTURING CORP.|Eastman Kodak ...,NaN
160,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class II),NaN,NaN,Fox Film Corp.,NaN
161,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class III),NaN,NaN,Electrical Research Products Inc.,NaN
162,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class III),NaN,NaN,Rko Radio Pictures,NaN
163,1930/31,SCIENTIFIC OR TECHNICAL AWARD (Class III),NaN,NaN,Rca-Photophone Inc.,NaN
209,1931/32,SCIENTIFIC OR TECHNICAL AWARD (Class II),NaN,NaN,Technicolor Motion Picture Corp.,NaN
210,1931/32,SCIENTIFIC OR TECHNICAL AWARD (Class III),NaN,NaN,Eastman Kodak Company,NaN
273,1932/33,SCIENTIFIC OR TECHNICAL AWARD (Class II),NaN,NaN,Electrical Research Products,NaN
274,1932/33,SCIENTIFIC OR TECHNICAL AWARD (Class II),NaN,NaN,Rca-Victor Company Inc.,NaN


In [259]:
missing_ids_by_category = (
    missing_nominee_ids["Category"]
    .value_counts()
)

display(missing_ids_by_category.head(20))

Category
FOREIGN LANGUAGE FILM                                                310
SCIENTIFIC AND TECHNICAL AWARD (Technical Achievement Award)         121
SCIENTIFIC OR TECHNICAL AWARD (Class III)                            116
SCIENTIFIC AND TECHNICAL AWARD (Scientific and Engineering Award)     89
SCIENTIFIC OR TECHNICAL AWARD (Class II)                              63
SCIENTIFIC OR TECHNICAL AWARD (Technical Achievement Award)           51
SCIENTIFIC OR TECHNICAL AWARD (Scientific and Engineering Award)      42
INTERNATIONAL FEATURE FILM                                            35
SCIENTIFIC AND TECHNICAL AWARD (Academy Award of Merit)               13
SCIENTIFIC OR TECHNICAL AWARD (Class I)                               11
SCIENTIFIC OR TECHNICAL AWARD (Academy Award of Merit)                 8
HONORARY FOREIGN LANGUAGE FILM AWARD                                   5
MEDAL OF COMMENDATION                                                  4
GORDON E. SAWYER AWARD                    

In [260]:
duplicate_count = oscars_cleaned.duplicated().sum()

print(f"Duplicate records: {duplicate_count:,}")

Duplicate records: 0


In [261]:
year_values = oscars_cleaned["Year"].astype(str)

year_formats = year_values.str.fullmatch(r"\d{4}").value_counts()

print("Year formats:")
print(year_formats)

print("\nYears with nonstandard formats:")
display(
    oscars_cleaned.loc[
        ~year_values.str.fullmatch(r"\d{4}"),
        ["Ceremony", "Year"]
    ].drop_duplicates()
)

Year formats:
Year
True     11861
False      276
Name: count, dtype: int64

Years with nonstandard formats:


,Ceremony,Year
0,1,1927/28
34,2,1928/29
72,3,1929/30
114,4,1930/31
164,5,1931/32
211,6,1932/33


In [262]:
oscars_cleaned["Year"] = (
    oscars_cleaned["Year"]
    .str.replace(r"^\d{2}(\d{2})/(\d{2})$", r"19\2", regex=True)
    .astype(int)
)

display(
    oscars_cleaned[["Ceremony", "Year"]]
    .drop_duplicates()
    .head(10)
)

,Ceremony,Year
0,1,1928
34,2,1929
72,3,1930
114,4,1931
164,5,1932
211,6,1933
276,7,1934
343,8,1935
437,9,1936
551,10,1937


In [263]:
category_columns = ["Class", "CanonicalCategory", "Category"]

display(oscars_cleaned[category_columns].nunique())

display(
    oscars_cleaned[category_columns]
    .drop_duplicates()
    .sort_values("CanonicalCategory")
    .head(30)
)

Class                  8
CanonicalCategory     66
Category             130
dtype: int64

,Class,CanonicalCategory,Category
0,Acting,ACTOR IN A LEADING ROLE,ACTOR
5676,Acting,ACTOR IN A LEADING ROLE,ACTOR IN A LEADING ROLE
442,Acting,ACTOR IN A SUPPORTING ROLE,ACTOR IN A SUPPORTING ROLE
2,Acting,ACTRESS IN A LEADING ROLE,ACTRESS
5686,Acting,ACTRESS IN A LEADING ROLE,ACTRESS IN A LEADING ROLE
452,Acting,ACTRESS IN A SUPPORTING ROLE,ACTRESS IN A SUPPORTING ROLE
8805,Title,ANIMATED FEATURE FILM,ANIMATED FEATURE FILM
5,Production,ART DIRECTION,ART DIRECTION
10293,Production,ART DIRECTION,PRODUCTION DESIGN
997,Production,ART DIRECTION (Black-and-White),ART DIRECTION (Black-and-White)


In [264]:
category_mapping = (
    oscars_cleaned.groupby("Category")["CanonicalCategory"]
    .nunique()
)

inconsistent_categories = category_mapping[category_mapping > 1]

print(f"Categories with multiple canonical mappings: {len(inconsistent_categories)}")

display(inconsistent_categories)

Categories with multiple canonical mappings: 0


Series([], Name: CanonicalCategory, dtype: int64)

In [265]:
text_columns = [
    "Film",
    "Name",
    "Nominees",
    "CanonicalCategory",
    "Category"
]

for column in text_columns:
    values = oscars_cleaned[column].dropna()
    extra_spaces = (values != values.str.strip()).sum()

    print(f"{column}: {extra_spaces} values with extra spaces")

Film: 0 values with extra spaces
Name: 0 values with extra spaces
Nominees: 0 values with extra spaces
CanonicalCategory: 0 values with extra spaces
Category: 0 values with extra spaces


In [266]:
supplemental_columns = ["Detail", "Note", "Citation"]

for column in supplemental_columns:
    print(f"\n{column}")
    print(f"Non-missing values: {oscars_cleaned[column].notna().sum():,}")

    display(
        oscars_cleaned[column]
        .dropna()
        .drop_duplicates()
        .head(5)
    )


Detail
Non-missing values: 3,187


0                 Nickie Elkins|The Patent Leather Kid
1    General Dolgorucki [Grand Duke Sergius Alexand...
2                                         Mrs. Pleznik
3                                Diane|Angela|The Wife
4                                       Sadie Thompson
Name: Detail, dtype: object


Note
Non-missing values: 626


9     NOTE: For this awards year, awards were presen...
15    NOTE: This nomination was not associated with ...
17    NOTE: Though no specific titles were indicated...
30    NOTE: This award was not associated with any s...
33    NOTE: \"The Academy Board of Judges on merit a...
Name: Note, dtype: object


Citation
Non-missing values: 1,201


32     To Warner Bros., for producing The Jazz Singer...
33     To Charles Chaplin, for acting, writing, direc...
158    To ELECTRICAL RESEARCH PRODUCTS, INC., RCA-PHO...
159    To DuPONT FILM MANUFACTURING CORP. and EASTMAN...
160    To FOX FILM CORP. for effective use of synchro...
Name: Citation, dtype: object

In [267]:
oscars_cleaned = oscars_cleaned.drop(
    columns=["Note", "Citation", "Detail"]
)

display(oscars_cleaned.head())

,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,NomineeIds,Winner
0,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Noose|The Patent Leather Kid,tt0019217|tt0018253,Richard Barthelmess,Richard Barthelmess,nm0001932,False
1,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Last Command|The Way of All Flesh,tt0019071|tt0019553,Emil Jannings,Emil Jannings,nm0417837,True
2,1,1928,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,nm0237571,False
3,1,1928,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,7th Heaven|Street Angel|Sunrise,tt0018379|tt0019429|tt0018455,Janet Gaynor,Janet Gaynor,nm0310980,True
4,1,1928,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,Sadie Thompson,tt0019344,Gloria Swanson,Gloria Swanson,nm0841797,False


In [268]:
winner_counts = (
    oscars_cleaned
    .groupby(["Ceremony", "Category"])["Winner"]
    .sum()
)

print(f"Groups with no winner: {(winner_counts == 0).sum()}")
print(f"Groups with one winner: {(winner_counts == 1).sum()}")
print(f"Groups with multiple winners: {(winner_counts > 1).sum()}")

display(winner_counts[winner_counts == 0].head(15))

Groups with no winner: 0
Groups with one winner: 2433
Groups with multiple winners: 239


Series([], Name: Winner, dtype: int64)

In [269]:
multiple_winner_groups = winner_counts[winner_counts > 1]

multiple_winner_categories = (
    multiple_winner_groups
    .reset_index(name="WinnerCount")
    .groupby("Category")
    .agg(
        Groups=("WinnerCount", "count"),
        MaxWinners=("WinnerCount", "max")
    )
    .sort_values("Groups", ascending=False)
)

display(multiple_winner_categories.head(20))

,Groups,MaxWinners
Category,,
HONORARY AWARD,39,6
SCIENTIFIC OR TECHNICAL AWARD (Class III),39,10
SCIENTIFIC AND TECHNICAL AWARD (Scientific and Engineering Award),31,14
SCIENTIFIC AND TECHNICAL AWARD (Technical Achievement Award),30,20
SCIENTIFIC OR TECHNICAL AWARD (Class II),28,7
SCIENTIFIC OR TECHNICAL AWARD (Technical Achievement Award),14,11
SPECIAL AWARD,14,6
SCIENTIFIC OR TECHNICAL AWARD (Scientific and Engineering Award),12,9
SCIENTIFIC AND TECHNICAL AWARD (Academy Award of Merit),5,2


In [270]:
print(f"Rows: {oscars_cleaned.shape[0]:,}")
print(f"Columns: {oscars_cleaned.shape[1]}")

print("\nData types:")
display(oscars_cleaned.dtypes)

print("\nMissing values:")
display(oscars_cleaned.isna().sum())

print("\nSample records:")
display(oscars_cleaned.head())

Rows: 12,137
Columns: 11

Data types:


Ceremony              int64
Year                  int64
Class                object
CanonicalCategory    object
Category             object
Film                 object
FilmId               object
Name                 object
Nominees             object
NomineeIds           object
Winner                 bool
dtype: object


Missing values:


Ceremony                0
Year                    0
Class                   0
CanonicalCategory       0
Category                0
Film                 1280
FilmId               1281
Name                 1203
Nominees              358
NomineeIds            885
Winner                  0
dtype: int64


Sample records:


,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,NomineeIds,Winner
0,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Noose|The Patent Leather Kid,tt0019217|tt0018253,Richard Barthelmess,Richard Barthelmess,nm0001932,False
1,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Last Command|The Way of All Flesh,tt0019071|tt0019553,Emil Jannings,Emil Jannings,nm0417837,True
2,1,1928,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,nm0237571,False
3,1,1928,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,7th Heaven|Street Angel|Sunrise,tt0018379|tt0019429|tt0018455,Janet Gaynor,Janet Gaynor,nm0310980,True
4,1,1928,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,Sadie Thompson,tt0019344,Gloria Swanson,Gloria Swanson,nm0841797,False


In [299]:
text_columns = oscars_cleaned.select_dtypes(include="object").columns

oscars_cleaned[text_columns] = oscars_cleaned[text_columns].astype("string")

oscars_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12137 entries, 0 to 12136
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Ceremony           12137 non-null  int64 
 1   Year               12137 non-null  int64 
 2   Class              12137 non-null  string
 3   CanonicalCategory  12137 non-null  string
 4   Category           12137 non-null  string
 5   Film               10857 non-null  string
 6   FilmId             10856 non-null  string
 7   Name               10934 non-null  string
 8   Nominees           11779 non-null  string
 9   NomineeIds         11252 non-null  string
 10  Winner             12137 non-null  bool  
dtypes: bool(1), int64(2), string(8)
memory usage: 960.2 KB


In [300]:
save_cleaned_data(oscars_cleaned, "oscars_cleaned.csv")

Saved to ../data/processed/oscars_cleaned.csv


#### Oscars Data Report

Original dataset: 12,137 records and 14 columns.

- **Removed columns:** `Note` (94.84% missing), `Citation` (90.10% missing), and `Detail` (73.74% missing) because they contained mostly missing data and were not essential to our analysis.
- **Removed rows:** None. We preserved records with missing film titles or IDs because some awards recognize individuals or organizations rather than movies.
- **Updated data types:** Converted `Winner` to boolean, replacing missing values with `False`. Converted `Year` to integer, using the ending year for historical ranges (e.g., `1927/28` to `1928`), and text columns to strings.
- **Checked duplicates:** 0 duplicate records found.
- **Validated:** Category mappings were consistent, and records containing multiple films had matching numbers of film titles and IDs.

Result: Reduced the dataset from 14 to 11 columns while preserving all 12,137 records.


In [298]:
oscars_cleaned.info()
oscars_cleaned.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12137 entries, 0 to 12136
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Ceremony           12137 non-null  int64 
 1   Year               12137 non-null  int64 
 2   Class              12137 non-null  object
 3   CanonicalCategory  12137 non-null  object
 4   Category           12137 non-null  object
 5   Film               10857 non-null  object
 6   FilmId             10856 non-null  object
 7   Name               10934 non-null  object
 8   Nominees           11779 non-null  object
 9   NomineeIds         11252 non-null  object
 10  Winner             12137 non-null  bool  
dtypes: bool(1), int64(2), object(8)
memory usage: 960.2+ KB


,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,NomineeIds,Winner
0,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Noose|The Patent Leather Kid,tt0019217|tt0018253,Richard Barthelmess,Richard Barthelmess,nm0001932,False
1,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Last Command|The Way of All Flesh,tt0019071|tt0019553,Emil Jannings,Emil Jannings,nm0417837,True
2,1,1928,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,nm0237571,False
3,1,1928,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,7th Heaven|Street Angel|Sunrise,tt0018379|tt0019429|tt0018455,Janet Gaynor,Janet Gaynor,nm0310980,True
4,1,1928,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,Sadie Thompson,tt0019344,Gloria Swanson,Gloria Swanson,nm0841797,False


#### imdb data

In [272]:
imdb = pd.read_csv("../data/raw/imdb_movies.csv")

imdb_cleaned = imdb.copy()

inspect_dataset(imdb_cleaned)

Rows: 759,954
Columns: 9

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 759954 entries, 0 to 759953
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   tconst          759954 non-null  object 
 1   titleType       759954 non-null  object 
 2   primaryTitle    759951 non-null  object 
 3   originalTitle   759951 non-null  object 
 4   isAdult         759954 non-null  int64  
 5   startYear       647396 non-null  float64
 6   endYear         0 non-null       float64
 7   runtimeMinutes  489537 non-null  float64
 8   genres          682409 non-null  object 
dtypes: float64(3), int64(1), object(5)
memory usage: 52.2+ MB

First five rows:


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000009,movie,Miss Jerry,Miss Jerry,0,1894.0,NaN,45.0,Romance
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,0,1897.0,NaN,100.0,"Documentary,News,Sport"
2,tt0000502,movie,Bohemios,Bohemios,0,1905.0,NaN,100.0,NaN
3,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906.0,NaN,70.0,"Action,Adventure,Biography"
4,tt0000591,movie,The Prodigal Son,L'enfant prodigue,0,1907.0,NaN,90.0,Drama


In [273]:
display(summarize_missing_values(imdb_cleaned))

,missing_count,missing_percentage
endYear,759954,100.00
runtimeMinutes,270417,35.58
startYear,112558,14.81
genres,77545,10.20
tconst,0,0.00
isAdult,0,0.00
originalTitle,3,0.00
primaryTitle,3,0.00
titleType,0,0.00


In [274]:
imdb_cleaned = imdb_cleaned.drop(columns=["endYear"])

display(imdb_cleaned.head())

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres
0,tt0000009,movie,Miss Jerry,Miss Jerry,0,1894.0,45.0,Romance
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,0,1897.0,100.0,"Documentary,News,Sport"
2,tt0000502,movie,Bohemios,Bohemios,0,1905.0,100.0,NaN
3,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906.0,70.0,"Action,Adventure,Biography"
4,tt0000591,movie,The Prodigal Son,L'enfant prodigue,0,1907.0,90.0,Drama


In [275]:
display(imdb_cleaned["titleType"].value_counts())

titleType
movie    759954
Name: count, dtype: int64

In [276]:
imdb_cleaned = imdb_cleaned.drop(columns=["titleType"])

In [277]:
display(imdb_cleaned["isAdult"].value_counts(dropna=False))

isAdult
0    750856
1      9098
Name: count, dtype: int64

In [278]:
imdb_cleaned["isAdult"] = imdb_cleaned["isAdult"].astype(bool)

In [279]:
display(imdb_cleaned["isAdult"].value_counts(dropna=False))

isAdult
False    750856
True       9098
Name: count, dtype: int64

In [280]:
missing_years = imdb_cleaned[imdb_cleaned["startYear"].isna()]

print(f"Movies missing release year: {len(missing_years):,}")

display(
    missing_years[
        ["tconst", "primaryTitle", "originalTitle", "runtimeMinutes", "genres"]
    ].head(10)
)

Movies missing release year: 112,558


,tconst,primaryTitle,originalTitle,runtimeMinutes,genres
46150,tt0063362,Las noches del Hombre Lobo,Las noches del Hombre Lobo,82.0,Horror
66149,tt0093025,Tales of the Brothers Quay,The Films of the Brothers Quay,78.0,"Animation,Documentary"
77472,tt0111597,The Veiled Hope,The Veiled Hope,52.0,Documentary
80909,tt0118530,18,18,NaN,Romance
82467,tt0120892,Winding Stair,Winding Stair,NaN,Thriller
82489,tt0121062,Alexandru Lapusneanu,Alexandru Lapusneanu,NaN,History
82534,tt0121158,Complica,Complica,NaN,NaN
82538,tt0121172,Cuo zai cuo,Cuo zai cuo,NaN,NaN
82539,tt0121173,The Curly-Headed Dude,The Curly-Headed Dude,NaN,NaN
82549,tt0121191,Deadly Confessions,Deadly Confessions,NaN,NaN


In [281]:
imdb_cleaned = imdb_cleaned.dropna(
    subset=["startYear", "genres", "runtimeMinutes"],
    how="all"
).copy()

imdb_cleaned["startYear"] = imdb_cleaned["startYear"].astype("Int64")
imdb_cleaned["runtimeMinutes"] = imdb_cleaned["runtimeMinutes"].astype("Int64")

print(f"Remaining movies: {len(imdb_cleaned):,}")

print("\nMissing values:")
display(
    imdb_cleaned[["startYear", "genres", "runtimeMinutes"]]
    .isna()
    .sum()
)

Remaining movies: 753,415

Missing values:


startYear         106019
genres             71006
runtimeMinutes    263878
dtype: int64

In [282]:
imdb_cleaned.head(20)

,tconst,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres
0,tt0000009,Miss Jerry,Miss Jerry,False,1894,45,Romance
1,tt0000147,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,False,1897,100,"Documentary,News,Sport"
2,tt0000502,Bohemios,Bohemios,False,1905,100,NaN
3,tt0000574,The Story of the Kelly Gang,The Story of the Kelly Gang,False,1906,70,"Action,Adventure,Biography"
4,tt0000591,The Prodigal Son,L'enfant prodigue,False,1907,90,Drama
5,tt0000615,Robbery Under Arms,Robbery Under Arms,False,1907,<NA>,Drama
6,tt0000630,Hamlet,Amleto,False,1908,<NA>,Drama
7,tt0000675,Don Quijote,Don Quijote,False,1908,<NA>,Drama
8,tt0000679,The Fairylogue and Radio-Plays,The Fairylogue and Radio-Plays,False,1908,120,"Adventure,Fantasy"
9,tt0000838,A Cultura do Cacau,A Cultura do Cacau,False,1909,<NA>,NaN


In [283]:
missing_runtimes = imdb_cleaned[
    imdb_cleaned["runtimeMinutes"].isna()
]

print(f"Movies missing runtime: {len(missing_runtimes):,}")
print(f"Percentage: {len(missing_runtimes) / len(imdb_cleaned) * 100:.2f}%")

display(
    missing_runtimes.head(10)
)

Movies missing runtime: 263,878
Percentage: 35.02%


,tconst,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres
5,tt0000615,Robbery Under Arms,Robbery Under Arms,False,1907,<NA>,Drama
6,tt0000630,Hamlet,Amleto,False,1908,<NA>,Drama
7,tt0000675,Don Quijote,Don Quijote,False,1908,<NA>,Drama
9,tt0000838,A Cultura do Cacau,A Cultura do Cacau,False,1909,<NA>,NaN
10,tt0000842,De Garraf a Barcelona,De Garraf a Barcelona,False,1909,<NA>,NaN
11,tt0000846,Un día en Xochimilco,Un día en Xochimilco,False,1909,<NA>,NaN
12,tt0000850,Los dos hermanos,Los dos hermanos,False,1909,<NA>,NaN
13,tt0000859,Fabricación del corcho en Sant Feliu de Guixols,Fabricación del corcho en Sant Feliu de Guixols,False,1909,<NA>,NaN
14,tt0000862,Faldgruben,Faldgruben,False,1909,<NA>,NaN
15,tt0000867,Fiesta de toros,Fiesta de toros,False,1909,<NA>,NaN


In [284]:
imdb_cleaned = imdb_cleaned.dropna(
    subset=["runtimeMinutes", "genres"],
    how="all"
).copy()

print(f"Remaining movies: {len(imdb_cleaned):,}")
display(summarize_missing_values(imdb_cleaned))

Remaining movies: 709,721


,missing_count,missing_percentage
runtimeMinutes,220184,31.02
startYear,106019,14.94
genres,27312,3.85
originalTitle,3,0.00
primaryTitle,3,0.00
tconst,0,0.00
isAdult,0,0.00


In [285]:
imdb_cleaned.head(20)

,tconst,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres
0,tt0000009,Miss Jerry,Miss Jerry,False,1894,45,Romance
1,tt0000147,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,False,1897,100,"Documentary,News,Sport"
2,tt0000502,Bohemios,Bohemios,False,1905,100,NaN
3,tt0000574,The Story of the Kelly Gang,The Story of the Kelly Gang,False,1906,70,"Action,Adventure,Biography"
4,tt0000591,The Prodigal Son,L'enfant prodigue,False,1907,90,Drama
5,tt0000615,Robbery Under Arms,Robbery Under Arms,False,1907,<NA>,Drama
6,tt0000630,Hamlet,Amleto,False,1908,<NA>,Drama
7,tt0000675,Don Quijote,Don Quijote,False,1908,<NA>,Drama
8,tt0000679,The Fairylogue and Radio-Plays,The Fairylogue and Radio-Plays,False,1908,120,"Adventure,Fantasy"
19,tt0000886,"Hamlet, Prince of Denmark",Hamlet,False,1910,<NA>,Drama


In [286]:
duplicate_ids = imdb_cleaned[
    imdb_cleaned.duplicated(subset=["tconst"], keep=False)
]

print(f"Duplicate IMDb IDs: {imdb_cleaned['tconst'].duplicated().sum():,}")
print(f"Records with duplicate IDs: {len(duplicate_ids):,}")

display(duplicate_ids.head(10))

Duplicate IMDb IDs: 0
Records with duplicate IDs: 0


,tconst,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres


In [287]:
print(imdb_cleaned.dtypes)

tconst            object
primaryTitle      object
originalTitle     object
isAdult             bool
startYear          Int64
runtimeMinutes     Int64
genres            object
dtype: object


In [288]:
text_columns = imdb_cleaned.select_dtypes(include="object").columns

imdb_cleaned[text_columns] = imdb_cleaned[text_columns].astype("string")

print(imdb_cleaned.dtypes)

tconst            string[python]
primaryTitle      string[python]
originalTitle     string[python]
isAdult                     bool
startYear                  Int64
runtimeMinutes             Int64
genres            string[python]
dtype: object


In [289]:
display(imdb_cleaned[["startYear", "runtimeMinutes"]].describe())

print("Invalid release years:",
      (imdb_cleaned["startYear"] < 1888).sum())

print("Zero or negative runtimes:",
      (imdb_cleaned["runtimeMinutes"] <= 0).sum())

,startYear,runtimeMinutes
count,603702.0,489537.0
mean,1997.488913,89.890315
std,28.468032,113.343078
min,1894.0,1.0
25%,1982.0,73.0
50%,2010.0,89.0
75%,2019.0,100.0
max,2032.0,51420.0


Invalid release years: 0
Zero or negative runtimes: 0


In [290]:
# Inspect movies with the longest runtimes
display(
    imdb_cleaned.nlargest(15, "runtimeMinutes")[
        ["tconst", "primaryTitle", "startYear", "runtimeMinutes", "genres"]
    ]
)

,tconst,primaryTitle,startYear,runtimeMinutes,genres
730444,tt8273150,Logistics,2012,51420,Documentary
581365,tt3854496,Ambiancé,2020,43200,Documentary
275113,tt12277054,Carnets Filmés (Liste Complète),2019,28643,Documentary
446464,tt2659636,Modern Times Forever,2011,14400,Documentary
427445,tt2355497,Beijing 2003,2004,9000,Documentary
353731,tt16226132,Soldier,2014,7200,<NA>
655090,tt5068890,Hunger!,2015,6000,"Documentary,Drama"
371378,tt1806963,Matrjoschka,2006,5700,Drama
657342,tt5136218,London EC1,2015,5460,"Comedy,Drama,Mystery"
155640,tt0284020,The Cure for Insomnia,1987,5220,"Documentary,Music"


In [291]:
for threshold in [180, 300, 600, 1440]:
    count = (imdb_cleaned["runtimeMinutes"] > threshold).sum()
    print(f"Movies longer than {threshold} minutes: {count:,}")

Movies longer than 180 minutes: 3,047
Movies longer than 300 minutes: 566
Movies longer than 600 minutes: 109
Movies longer than 1440 minutes: 31


In [292]:
future_movies = imdb_cleaned[
    imdb_cleaned["startYear"] > 2026
]

print(f"Movies with future release years: {len(future_movies):,}")

display(
    future_movies[
        ["tconst", "primaryTitle", "startYear", "runtimeMinutes", "genres"]
    ].head(15)
)

Movies with future release years: 1,522


,tconst,primaryTitle,startYear,runtimeMinutes,genres
196060,tt0418104,Rip Cage,2027,<NA>,"Horror,Mystery,Sci-Fi"
231792,tt10303934,Asesina Aprendiz,2027,<NA>,"Comedy,Crime"
232209,tt10324136,Brawler,2027,<NA>,Action
232847,tt10355586,Silent Bark,2027,<NA>,"Comedy,Crime,Drama"
233611,tt10385020,"Blood, Skulls and Chrome",2027,<NA>,"Action,Crime"
236268,tt10477440,Those Fabulous Places,2027,<NA>,"Comedy,Drama,Romance"
239423,tt10614152,Naked (Inside the box),2027,<NA>,Thriller
240083,tt10647654,Bristol,2027,110,"Drama,Sci-Fi"
244495,tt10828378,Second Nature,2027,<NA>,"Drama,Mystery,Thriller"
244778,tt10843622,Partum,2027,<NA>,"Drama,Sci-Fi,Thriller"


In [293]:
imdb_cleaned = imdb_cleaned[
    imdb_cleaned["startYear"].isna() |
    (imdb_cleaned["startYear"] <= 2026)
].copy()

print(f"Remaining movies: {len(imdb_cleaned):,}")
print(f"Future releases: {(imdb_cleaned['startYear'] > 2026).sum():,}")

Remaining movies: 708,199
Future releases: 0


##### IMDb ratings

The ratings file covers all IMDb titles, not only movies. A left join keeps every movie, so movies
without a rating get missing values for `averageRating` and `numVotes`.

*Added with Claude Code (AI-generated).*

In [ ]:
imdb_ratings = pd.read_csv(raw_data_directory / "imdb_ratings.csv")

print(f"Rating records: {len(imdb_ratings):,}")
print(f"Duplicate IMDb IDs: {imdb_ratings['tconst'].duplicated().sum():,}")
display(imdb_ratings[["averageRating", "numVotes"]].describe())

In [ ]:
imdb_cleaned = imdb_cleaned.merge(imdb_ratings, on="tconst", how="left")
imdb_cleaned["numVotes"] = imdb_cleaned["numVotes"].astype("Int64")

rated_count = imdb_cleaned["averageRating"].notna().sum()
print(f"Movies with a rating: {rated_count:,} of {len(imdb_cleaned):,}")
print(f"Movies with at least 1,000 votes: {(imdb_cleaned['numVotes'] >= 1000).sum():,}")

In [294]:
inspect_dataset(imdb_cleaned)

display(summarize_missing_values(imdb_cleaned))

Rows: 708,199
Columns: 7

Dataset information:
<class 'pandas.core.frame.DataFrame'>
Index: 708199 entries, 0 to 759953
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   tconst          708199 non-null  string
 1   primaryTitle    708196 non-null  string
 2   originalTitle   708196 non-null  string
 3   isAdult         708199 non-null  bool  
 4   startYear       602180 non-null  Int64 
 5   runtimeMinutes  489193 non-null  Int64 
 6   genres          680894 non-null  string
dtypes: Int64(2), bool(1), string(4)
memory usage: 39.8 MB

First five rows:


,tconst,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres
0,tt0000009,Miss Jerry,Miss Jerry,False,1894,45,Romance
1,tt0000147,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,False,1897,100,"Documentary,News,Sport"
2,tt0000502,Bohemios,Bohemios,False,1905,100,<NA>
3,tt0000574,The Story of the Kelly Gang,The Story of the Kelly Gang,False,1906,70,"Action,Adventure,Biography"
4,tt0000591,The Prodigal Son,L'enfant prodigue,False,1907,90,Drama


,missing_count,missing_percentage
runtimeMinutes,219006,30.92
startYear,106019,14.97
genres,27305,3.86
originalTitle,3,0.00
primaryTitle,3,0.00
tconst,0,0.00
isAdult,0,0.00


In [295]:
save_cleaned_data(imdb_cleaned, "imdb_cleaned.csv")

Saved to ../data/processed/imdb_cleaned.csv


#### IMDb Data Report
Original dataset: 759,954 movies and 9 columns.

- **Removed columns:** `endYear` (100% missing) and `titleType` (100% of records were movies, redundant).
- **Removed rows:** Movies missing both `genres` and `runtimeMinutes` because they lacked information for our analysis. We also removed movies with release years after 2026 since they have not been released yet and their information may change. We preserved records with useful genre or runtime information.
- **Updated data types:** Converted `isAdult` to boolean, `startYear` and `runtimeMinutes` to nullable integers, and text columns to strings.
- **Checked duplicates:** 0 duplicate IMDb IDs found.
- **Added ratings:** Joined `averageRating` and `numVotes` from IMDb's separate ratings file on `tconst`. About half of movies have no rating, and we kept them with missing values.
- **Runtime outliers:** Only 109 movies exceeded 10 hours and 31 exceeded 24 hours. We kept them because they may be legitimate films.

Result: Reduced the dataset from 9 to 7 columns, then added 2 rating columns, while preserving useful movie information, 708,199 movies.


In [297]:
imdb_cleaned.info()
imdb_cleaned.head()

<class 'pandas.core.frame.DataFrame'>
Index: 708199 entries, 0 to 759953
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   tconst          708199 non-null  string
 1   primaryTitle    708196 non-null  string
 2   originalTitle   708196 non-null  string
 3   isAdult         708199 non-null  bool  
 4   startYear       602180 non-null  Int64 
 5   runtimeMinutes  489193 non-null  Int64 
 6   genres          680894 non-null  string
dtypes: Int64(2), bool(1), string(4)
memory usage: 39.8 MB


,tconst,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres
0,tt0000009,Miss Jerry,Miss Jerry,False,1894,45,Romance
1,tt0000147,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,False,1897,100,"Documentary,News,Sport"
2,tt0000502,Bohemios,Bohemios,False,1905,100,<NA>
3,tt0000574,The Story of the Kelly Gang,The Story of the Kelly Gang,False,1906,70,"Action,Adventure,Biography"
4,tt0000591,The Prodigal Son,L'enfant prodigue,False,1907,90,Drama
